# 07 — Parameter Study

**Topic**: Sensitivity analysis — loop over $k_3$ and damping, bifurcation-style diagram.

**Reference**: Krack & Gross (2019) §5.1

**Estimated runtime**: < 45 seconds

## Theory

For the Duffing oscillator (K&G §5.1), the hardening effect of $k_3 > 0$ shifts the resonance frequency upward:

$$\omega_{res} \approx \sqrt{k/m + \frac{3}{4}k_3 A^2 / m} \tag{1}$$

where $A$ is the response amplitude. This is the **backbone approximation** from the method of averaging.

As $k_3$ increases:
- The resonance frequency increases (hardening trend).
- The FRF peak bends more strongly to the right.
- For large $k_3$, the system develops fold points (jump phenomena) even at low forcing levels.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib
matplotlib.use('Agg')

import numpy as np
import matplotlib.pyplot as plt

from nlvib.nonlinearities.elements import cubic_spring
from nlvib.systems.oscillators import SingleMassOscillator
from nlvib.solvers.harmonic_balance import hb_residual
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions

m, d_base, k = 1.0, 0.02, 1.0
F_amp = 0.1
H = 3
excitation = {'dof': 0, 'amplitude': F_amp, 'harmonic': 1}

print("Setup complete.")

In [ ]:
# Sweep over k3 values — compute peak amplitude and resonance frequency
k3_values = [0.0, 0.1, 0.5, 1.0, 2.0]  # ← try changing this array

def run_frf_for_k3(k3_val, damping=0.02):
    """Run continuation for given k3, return (peak_omega, peak_amp)."""
    sys_ = SingleMassOscillator(m=m, d=damping, k=k)
    if k3_val > 0:
        sys_.add_nonlinear_element(cubic_spring(k3=k3_val, dof_index=0))

    omega0 = 0.65
    Q0 = np.zeros(2*H + 1)
    Q0[1] = F_amp / abs(-(omega0**2)*m + k + 1j*omega0*damping)

    def res_fn(Q, lam):
        return hb_residual(Q, lam, sys_, H, excitation)

    opts = ContinuationOptions(
        ds_initial=0.03,
        ds_min=1e-5,
        ds_max=0.1,
        max_steps=400,
        newton_tol=1e-9,
        lambda_min=0.6,
        lambda_max=1.5,
    )
    result = ContinuationSolver().run(res_fn, Q0, omega0, opts)

    omega_a = result.solutions[:, -1]
    Q_a     = result.solutions[:, :-1]
    amp_a   = np.sqrt(Q_a[:, 1]**2 + Q_a[:, 2]**2)

    peak_idx = int(np.argmax(amp_a))
    return float(omega_a[peak_idx]), float(amp_a[peak_idx]), omega_a, amp_a, ~result.stability

print("Running parameter sweep over k3...")
results_k3 = {}
for k3_v in k3_values:
    peak_om, peak_a, om_arr, amp_arr, stab_arr = run_frf_for_k3(k3_v)
    results_k3[k3_v] = (peak_om, peak_a, om_arr, amp_arr, stab_arr)
    print(f"  k3={k3_v:.1f}: peak_omega={peak_om:.4f}, peak_amp={peak_a:.4f}")

In [ ]:
# Plot bifurcation-style: FRF overlaid for all k3, and peak_omega vs k3
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

cmap = plt.cm.viridis
colors = cmap(np.linspace(0.1, 0.9, len(k3_values)))

for (k3_v, color) in zip(k3_values, colors):
    peak_om, peak_a, om_arr, amp_arr, stab_arr = results_k3[k3_v]
    ax1.plot(om_arr, amp_arr, '-', color=color, lw=2, label=f'k3={k3_v}')
    ax1.plot(peak_om, peak_a, 'o', color=color, ms=8)

ax1.set_xlabel('$\\Omega$ (rad/s)')
ax1.set_ylabel('Amplitude')
ax1.set_title('Duffing FRF for varying $k_3$ — hardening trend')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

peak_omegas = [results_k3[k3_v][0] for k3_v in k3_values]
peak_amps   = [results_k3[k3_v][1] for k3_v in k3_values]
ax2.plot(k3_values, peak_omegas, 'o-', lw=2, color='tab:blue', label='Peak $\\Omega_{res}$')
ax2_r = ax2.twinx()
ax2_r.plot(k3_values, peak_amps, 's--', lw=2, color='tab:orange', label='Peak amplitude')
ax2.set_xlabel('$k_3$')
ax2.set_ylabel('Resonance frequency $\\Omega_{res}$ (rad/s)', color='tab:blue')
ax2_r.set_ylabel('Peak amplitude', color='tab:orange')
ax2.set_title('Bifurcation diagram: $\\Omega_{res}$ vs $k_3$')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
fig

In [ ]:
# Effect of damping ratio on FRF peak height
damping_values = [0.005, 0.01, 0.02, 0.05, 0.1]  # ← try changing this
k3_fixed = 0.5

print("Running damping sweep...")
peak_amps_damp = []
for d_val in damping_values:
    p_om, p_a, om_a, amp_a, stab_a = run_frf_for_k3(k3_fixed, damping=d_val)
    peak_amps_damp.append(p_a)
    print(f"  d={d_val:.3f}: peak_amp={p_a:.4f}")

# Also compute linear (k3=0) peak amplitude for reference: F/(d*omega_n)
omega_n = np.sqrt(k/m)
linear_peaks = [F_amp / (d_val * omega_n * m) for d_val in damping_values]

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(damping_values, peak_amps_damp, 'o-', lw=2, label=f'Nonlinear (k3={k3_fixed})')
ax.loglog(damping_values, linear_peaks, 's--', lw=2, label='Linear (k3=0)')
ax.set_xlabel('Damping coefficient $d$ (N·s/m)')
ax.set_ylabel('Peak amplitude')
ax.set_title('Effect of damping on FRF peak height')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
fig

## Key Takeaways

- Increasing $k_3$ shifts the resonance frequency upward and eventually causes fold points (jump phenomena).
- Peak amplitude decreases slightly with $k_3$ for fixed forcing — the nonlinearity distributes energy to higher harmonics.
- For the linear system, peak amplitude scales as $1/(d\omega_n)$ — higher damping linearly reduces the peak.
- The nonlinear system shows a shallower amplitude decay with damping at large amplitudes — amplitude-stiffening effect.